# Next Lab — Free Cloud CPU LoRA Training

This notebook runs a small CPU-only LoRA experiment in a free Google Colab runtime. It uses distilbert/distilgpt2 (about 82M parameters) and deliberately limits the first run to 25 steps.

**Cost guard:** keep the runtime on CPU. Do not select a paid GPU, paid Colab plan, or any paid service. Free runtime availability and duration are not guaranteed. CPU training may be slow.

## 1. Confirm CPU runtime

In Colab choose Runtime → Change runtime type → CPU (or leave the default CPU runtime). The next cell checks that CUDA is not used.

In [ ]:
!pip -q install "transformers>=4.45,<5" "datasets>=3,<4" "peft>=0.14,<1" accelerate

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert not torch.cuda.is_available(), "This notebook is CPU-only. Switch to a CPU runtime before continuing."
print("CPU threads:", os.cpu_count())

## 2. Upload your dataset

Upload a .jsonl file with one JSON object per line and a text field, or a .csv file with a text column. In Next Lab, download the selected dataset from the Training page first. Use only data you are authorized to process.

In [ ]:
from google.colab import files
uploaded = files.upload()
data_path = next(iter(uploaded))
print("Uploaded:", data_path)

In [ ]:
from datasets import load_dataset
if data_path.lower().endswith(".jsonl"):
    ds = load_dataset("json", data_files=data_path, split="train")
elif data_path.lower().endswith(".csv"):
    ds = load_dataset("csv", data_files=data_path, split="train")
else:
    raise ValueError("Please upload JSONL or CSV with a text field/column.")
if "text" not in ds.column_names:
    raise ValueError(f"Dataset needs a text field/column. Found: {ds.column_names}")
ds = ds.filter(lambda row: isinstance(row["text"], str) and bool(row["text"].strip()))
if len(ds) == 0:
    raise ValueError("No non-empty text rows were found.")
MAX_TRAIN_ROWS = 300
ds = ds.select(range(min(len(ds), MAX_TRAIN_ROWS)))
print("Training rows:", len(ds))
print("Sample:", ds[0]["text"][:250])

## 3. Configure the small CPU model

The initial CPU path is intentionally fixed to distilbert/distilgpt2. If your Next Lab model record uses a different base model, create a model record with this exact base model for this first experiment. Later we can add architecture-aware model support.

In [ ]:
BASE_MODEL = "distilbert/distilgpt2"
OUTPUT_DIR = "/content/next-lab-cpu-lora"
MAX_LENGTH = 128
EPOCHS = 1
MAX_STEPS = 25
BATCH_SIZE = 1
GRAD_ACCUM = 4

In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, TrainingArguments,
    Trainer, DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
model.to("cpu")
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=4,
    lora_alpha=8,
    lora_dropout=0.05,
    target_modules=["c_attn", "c_proj"],
    fan_in_fan_out=True,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Model device:", next(model.parameters()).device)

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

tokenized = ds.map(tokenize, batched=True, remove_columns=ds.column_names)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=1e-4,
    logging_steps=5,
    save_strategy="no",
    no_cuda=True,
    fp16=False,
    dataloader_num_workers=0,
    report_to="none",
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

## 4. Run the CPU training

This cell starts training. It is limited to 25 optimizer steps for the first experiment. CPU training may still take a while; a free notebook session can disconnect.

In [ ]:
train_result = trainer.train()
print(train_result.metrics)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("LoRA adapter saved:", OUTPUT_DIR)

## 5. Download the LoRA adapter

The ZIP contains adapter weights/configuration and tokenizer files. It is not a standalone full model; keep the base model ID distilbert/distilgpt2 with it.

In [ ]:
!cd /content && zip -qr next-lab-cpu-lora-adapter.zip next-lab-cpu-lora
from google.colab import files
files.download("/content/next-lab-cpu-lora-adapter.zip")

## Limitations

- This is a bounded CPU experiment, not full pretraining or production-scale fine-tuning.
- It does not automatically update the Next Lab training job or upload the adapter.
- Keep the training job in Next Lab as queued until a verified adapter-import workflow exists.
- Free CPU sessions can be slow, interrupted, or unavailable. No paid compute is started.